In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from datetime import datetime, timedelta

# Veriyi yükle (Phase 1: priority yok)
df = pd.read_csv('ER_Phase2_Group_13.csv')
df = df[['Patient_ID', 'Arrival_Clock', 'Service_Required_Min']].copy()

# Arrival_Clock → dakika (08:00 baz)
# Veriler gece yarısını geçtiği için monoton artan sırayla işliyoruz
base = datetime.strptime('08:00:00', '%H:%M:%S')

def clock_to_min(times):
    """Sıralı saat listesini gece yarısı geçişini de dikkate alarak dakikaya çevirir."""
    result = []
    day_offset = 0
    prev_sec = -1
    for t in times:
        dt = datetime.strptime(t, '%H:%M:%S')
        sec = dt.hour * 3600 + dt.minute * 60 + dt.second
        if sec < prev_sec:   # gece yarısı geçti
            day_offset += 86400
        prev_sec = sec
        total_sec = sec + day_offset
        base_sec  = base.hour * 3600
        result.append((total_sec - base_sec) / 60)
    return result

df['arrival_min'] = clock_to_min(df['Arrival_Clock'].tolist())
df['interarrival_min'] = df['arrival_min'].diff()

# --- Özet ---
print('=== VERİ ÖZET ===')
print(df.head(10).to_string(index=False))
print(f'\nToplam hasta : {len(df)}')
print(f'İlk geliş    : {df["Arrival_Clock"].iloc[0]}  ({df["arrival_min"].iloc[0]:.2f} dk)')
print(f'Son geliş    : {df["Arrival_Clock"].iloc[-1]}  ({df["arrival_min"].iloc[-1]:.2f} dk)')
print(f'Toplam süre  : {df["arrival_min"].iloc[-1] - df["arrival_min"].iloc[0]:.1f} dk')

print('\n=== SERVİS SÜRESİ İSTATİSTİKLERİ ===')
print(df['Service_Required_Min'].describe().round(4))

print('\n=== İNTER-ARRİVAL İSTATİSTİKLERİ ===')
print(df['interarrival_min'].describe().round(4))

# --- Temel parametreler ---
total_time = df['arrival_min'].iloc[-1] - df['arrival_min'].iloc[0]
lam = (len(df) - 1) / total_time          # hasta/dk
mu  = 1 / df['Service_Required_Min'].mean()  # hasta/dk (tek doktor)
s   = 5
rho = lam / (s * mu)
cv  = df['Service_Required_Min'].std() / df['Service_Required_Min'].mean()

print('\n=== TEMEL PARAMETRELER ===')
print(f'λ  (geliş hızı)         = {lam:.4f} hasta/dk  ({lam*60:.2f} hasta/saat)')
print(f'μ  (servis hızı/doktor) = {mu:.4f} hasta/dk   (ort. servis = {1/mu:.2f} dk)')
print(f's  (doktor sayısı)      = {s}')
print(f'ρ  (utilization)        = {rho:.4f}  ({rho*100:.1f}%)')
print(f'Cv (servis CV)          = {cv:.4f}')
